# 02 — Data Cleaning: Ajuntament BCN Tree Inventory

**Purpose:** Apply CRISP-DM Phase 3 (Data Preparation) cleaning operations
to the Barcelona street and park tree inventory before grid aggregation.
Documents every quality issue found, the treatment applied, the missingness
mechanism assumed, rows affected, and residual concerns.

**Inputs:**
- `data/arbrat-viari.csv` — 143,610 street trees (Ajuntament Open Data BCN)
- `data/arbrat-zona.csv` — 45,480 park trees
- `data/fungalroot.csv` — FungalRoot v2.0 mycorrhizal type lookup (14,919 species)
- `data/bcn-boundary.geojson` — municipal boundary for spatial extent validation

**Output:** `data/trees_cleaned.parquet` — deduplicated, typed, flagged, joinable

**Cleaning audit:** `phase-3/data-cleaning-report.md`
**Cleaning log:** `docs/data-cleaning-log.md`

**Course scaffold:** CRISP-DM Phase 3.2 (Chapman et al. 2000, p. 28) —
Select, Clean, Construct, Integrate, Format, Verify.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Deterministic seed ────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Paths (relative to repo root) ─────────────────────────────────────────
DATA = Path("data")

PATHS = {
    "street_trees":  DATA / "arbrat-viari.csv",
    "park_trees":    DATA / "arbrat-zona.csv",
    "fungalroot":    DATA / "fungalroot.csv",
    "boundary":      DATA / "bcn-boundary.geojson",
    "districts":     DATA / "bcn-districts.geojson",
    "out_parquet":   DATA / "trees_cleaned.parquet",
}

CRS_PROJ = "EPSG:25831"   # ETRS89 / UTM Zone 31N (metre-unit)

print("All paths configured.")
for name, p in PATHS.items():
    exists = p.exists()
    print(f"  {name:20s}  {'FOUND' if exists else 'MISSING':<8s}  {p}")

In [ ]:
# ── Load raw CSVs ─────────────────────────────────────────────────────────
viari_raw = pd.read_csv(PATHS["street_trees"], encoding="utf-8", low_memory=False)
zona_raw  = pd.read_csv(PATHS["park_trees"], encoding="utf-8", low_memory=False)

viari_raw["source"] = "street"
zona_raw["source"]  = "park"

df_raw = pd.concat([viari_raw, zona_raw], ignore_index=True)
df = df_raw.copy()

print(f"Street trees  : {len(viari_raw):>7,}")
print(f"Park trees    : {len(zona_raw):>7,}")
print(f"Combined raw  : {len(df_raw):>7,}")
print(f"Columns       : {len(df_raw.columns)}")
print()
df.head(3)

## Task 1 — Select

### Row selection

All 189,090 rows are retained. No temporal filter is applied because the
Ajuntament publishes a snapshot-state inventory — there is no versioned
time series to filter. The full municipal extent (10 districts, 73 barris)
is in scope.

### Column selection

36 of 43 columns are dropped because they are irrelevant to the mycorrhizal
question (tree address, irrigation type, Catalan/Castilian common names,
catalogation status, etc.). Retained columns are those needed for:
1. **Spatial join** to 400 m grid: `x_etrs89`, `y_etrs89`
2. **Species identification**: `cat_nom_cientific`, `cat_especie_id`
3. **District / barri aggregation**: `nom_districte`, `nom_barri`,
   `codi_districte`, `codi_barri`
4. **Colonisation-uncertainty proxy**: `data_plantacio`
5. **Inventory provenance**: `codi` (unique tree ID), `source` (street/park)
6. **Vegetation type context**: `tipus_element` (tree vs palm)

### Decision rationale (SEL-005 from data-cleaning-report)

| Dropped column group | Count | Reason |
|----------------------|-------|--------|
| Common names (cat, cast, cat) | 3 | Redundant with scientific name |
| Address / street name | 2 | Not needed — spatial coord is sufficient |
| Irrigation / water type | 2 | Not relevant to mycorrhizal type |
| Catalogation / protection status | 1 | Out of scope |
| Green space name | 1 | Not needed |
| Geometry WKT string | 1 | Reconstructed from x/y coords |
| Other metadata | 26 | Administrative fields |

In [ ]:
n_before = df.shape[1]

KEEP_COLS = [
    "codi",               # unique tree ID
    "x_etrs89",           # UTM31N X coordinate
    "y_etrs89",           # UTM31N Y coordinate
    "latitud",            # WGS-84 latitude (for quick sanity checks)
    "longitud",           # WGS-84 longitude
    "cat_nom_cientific",  # scientific species name
    "cat_especie_id",     # species ID code
    "data_plantacio",     # planting date (mostly null)
    "nom_districte",      # district name (Catalan)
    "nom_barri",          # neighbourhood name (Catalan)
    "codi_districte",     # district code
    "codi_barri",         # neighbourhood code
    "tipus_element",      # tree / palm / shrub type
    "source",             # street or park provenance
]

missing_cols = [c for c in KEEP_COLS if c not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found in raw data: {missing_cols}")

df = df[KEEP_COLS].copy()
n_after = df.shape[1]
n_dropped = n_before - n_after

print(f"Columns before selection : {n_before}")
print(f"Columns after selection  : {n_after}")
print(f"Columns dropped          : {n_dropped}  (SEL-005)")
print(f"Rows retained            : {len(df):,}")
print()
print("Retained columns:")
for c in df.columns:
    print(f"  {c:25s}  dtype={df[c].dtype}")

## Task 2 — Clean

### Issues and treatment plan

All issues identified in the Phase 3 data-quality audit
(`phase-3/data-cleaning-report.md`) and the consolidated cleaning log
(`docs/data-cleaning-log.md`) are addressed below:

| # | Issue | Severity | Treatment | Cell |
|---|-------|----------|-----------|------|
| A1 | Species name inconsistency (`Quercus ilex`, `Q. ilex`) | MODERATE | Normalise to lowercase `genus species` | 7 |
| A2 | Genus-only records (25 rows, 0.01%) | MINOR | Flag `genus_only=True`; fall back to genus-level FungalRoot | 9 |
| A3 | 45,272 trees (24%) outside MYCO_LOOKUP top-20 | MAJOR | Retain in full dataset; flag `myco_unresolved=True`; grid stats computed from matched subset only | 9 |
| A4 | 81% null `data_plantacio` | MINOR | Flag `colonisation_uncertain=True`; no imputation | 9 |
| A5 | 36 irrelevant columns | MINOR | Handled in Task 1 (SEL-005) | 5 |
| A6 | Duplicate rows from re-ingestion | MINOR | Deferred — not treated; no SHA-256 row hash computed | — |
| F3 | Myco type categories (11 raw) vs pipeline codes (AM/EM/NM) | MINOR | Normalise with `_normalise_myco()` helper | 10 |
| U1 | Sealed surface scale misread as 0-100 (BUG-3) | CRITICAL | Verified in raster notebook; scale=1.0 confirmed | — |

### Missingness classification

| Column | Null rate | Mechanism | Justification |
|--------|-----------|-----------|---------------|
| `cat_nom_cientific` | 0% | — | Complete |
| `cat_especie_id` | 0% | — | Complete |
| `data_plantacio` | 81.0% | MAR | Older trees less likely to have recorded planting date; missingness correlates with age |
| `nom_barri` / `codi_barri` | 0.01% | MCAR | 6 records with missing district — likely data-entry error |
| Species outside MYCO_LOOKUP | 24% | MAR | Rarer species less likely to have resolved mycorrhizal type in FungalRoot |

In [ ]:
# ── Parse planting dates ──────────────────────────────────────────────────
# Profiling showed 81% null. Among non-null, 28 are future-dated (after today)
# and 8 are pre-1900. All 35,914 non-null strings parse successfully.

df["data_plantacio_raw"] = df["data_plantacio"].astype(str)  # preserve before parse
df["plant_date"] = pd.to_datetime(df["data_plantacio"], errors="coerce")

n_raw_string   = df["data_plantacio"].notna().sum()
n_parsed       = df["plant_date"].notna().sum()
n_null         = df["data_plantacio"].isna().sum()
n_unparseable  = n_raw_string - n_parsed

now = pd.Timestamp("2026-05-26")
n_future       = (df["plant_date"] > now).sum()
n_pre1900      = ((df["plant_date"] < pd.Timestamp("1900-01-01"))
                  & df["plant_date"].notna()).sum()

print(f"Non-null date strings : {n_raw_string:>7,}")
print(f"  Parsed successfully  : {n_parsed:>7,}")
print(f"  Unparseable (→ null) : {n_unparseable:>7,}")
print(f"Null date strings     : {n_null:>7,}  ({n_null/len(df)*100:.1f}%)")
print()
print(f"Future-dated records  : {n_future:>7,}  (will be flagged)")
print(f"Pre-1900 records      : {n_pre1900:>7,}  (will be flagged)")

# Flag anomalous dates for traceability
df["plant_date_anomaly"] = ""
df.loc[df["plant_date"] > now, "plant_date_anomaly"] = "future"
df.loc[(df["plant_date"] < pd.Timestamp("1900-01-01"))
       & df["plant_date"].notna(), "plant_date_anomaly"] = "pre-1900"
# Cap anomalous dates to NaN for downstream use (we flag, not use)
df.loc[df["plant_date_anomaly"] != "", "plant_date"] = pd.NaT

n_capped = (df["plant_date_anomaly"] != "").sum()
print(f"Anomalous dates capped  : {n_capped:>7,}  (set to NaT, preserved in anomaly column)")

In [ ]:
# ── Type coercion ─────────────────────────────────────────────────────────
# x_etrs89 / y_etrs89: already float64 — confirm range is sensible
# cat_especie_id: already int64 — assert positive
# codi_districte, codi_barri: currently float (from CSV read) — cast to Int64 (nullable)

# Coordinate sanity: Barcelona UTM31N bounds
COORD_BOUNDS = {
    "x_etrs89": (420_000, 436_000),    # UTM X
    "y_etrs89": (4_574_000, 4_592_000), # UTM Y
    "latitud":  (41.30, 41.48),          # WGS-84
    "longitud": (2.04, 2.24),
}

assert df["x_etrs89"].dtype == np.float64
assert df["y_etrs89"].dtype == np.float64
assert df["x_etrs89"].between(*COORD_BOUNDS["x_etrs89"]).all(), "x out of BCN bounds"
assert df["y_etrs89"].between(*COORD_BOUNDS["y_etrs89"]).all(), "y out of BCN bounds"
print("Coordinate bounds: PASS")

# District codes: float with .0 values → Int64 (nullable integer)
df["codi_districte"] = pd.to_numeric(df["codi_districte"], errors="coerce")
df["codi_barri"]     = pd.to_numeric(df["codi_barri"], errors="coerce")

assert df["codi_districte"].dropna().between(1, 10).all(), "district code out of range"
print(f"District codes (1-10): PASS  (null: {df['codi_districte'].isna().sum()})")

# Species ID: already integer — quick range check
assert df["cat_especie_id"].min() > 0, "species_id must be positive"
print(f"Species IDs:         PASS  (range {df['cat_especie_id'].min()}–{df['cat_especie_id'].max()})")

# Categorical dtypes for low-cardinality string columns
df["source"] = df["source"].astype("category")
df["tipus_element"] = df["tipus_element"].astype("category")
print(f"Categorical columns encoded: source ({df['source'].nunique()}), tipus_element ({df['tipus_element'].nunique()})")

In [ ]:
# ── Missing values: four-strategy decision ────────────────────────────────
# Strategy 1: DROP rows where core identifier is missing (none in this dataset)
# Strategy 2: FLAG rows where data_plantacio is null (81%) — colonisation_uncertain
# Strategy 3: FLAG rows where species is not in MYCO_LOOKUP (24%) — myco_unresolved
# Strategy 4: FLAG genus-only records (25 rows) — genus_only

# ── Strategy 2: planting date null flag ────────────────────────────────────
df["plant_date_missing"] = df["data_plantacio"].isna()
print(f"Strategy 2 — plant_date_missing:"
      f" {df['plant_date_missing'].sum():,} rows  ({df['plant_date_missing'].mean()*100:.1f}%)")

# ── Strategy 4: genus-only flag ───────────────────────────────────────────
def is_genus_only(name: str) -> bool:
    """True if the scientific name is genus-only (e.g. 'Washingtonia sp')"""
    if not isinstance(name, str):
        return False
    n = name.strip()
    if "sp." in n or n.endswith(" sp"):
        return True
    parts = n.replace("\u00d7", " ").split()  # multiplication sign
    parts = [p for p in parts if p and p not in ("x", "\u00d7")]
    return len(parts) < 2

df["genus_only"] = df["cat_nom_cientific"].apply(is_genus_only)
n_genus = df["genus_only"].sum()
print(f"Strategy 4 — genus_only:"
      f" {n_genus:,} rows  ({n_genus/len(df)*100:.4f}%)")
if n_genus > 0:
    print(f"  Examples: {df[df['genus_only']]['cat_nom_cientific'].value_counts().to_dict()}")

# ── Nominal missing: fill district/barri with UNKNOWN ─────────────────────
for col in ["nom_districte", "nom_barri", "codi_districte"]:
    n_miss = df[col].isna().sum()
    if n_miss > 0:
        df[col] = df[col].fillna("UNKNOWN" if df[col].dtype == object else -1)
        print(f"  Filled {n_miss} nulls in {col} -> 'UNKNOWN'")

print("\nMissing values after cleaning:")
missing_after = df.isna().mean().sort_values(ascending=False)
print(missing_after[missing_after > 0].to_string())

In [ ]:
# ── Outliers: cap and flag, never drop ────────────────────────────────────
outlier_flags = pd.Series(False, index=df.index)

# 1. Coordinate outliers: none found in profiling, but defensive check
coord_oo = ((df["latitud"] < 41.30) | (df["latitud"] > 41.48) |
            (df["longitud"] < 2.04) | (df["longitud"] > 2.24))
if coord_oo.any():
    print(f"WARNING: {coord_oo.sum()} coordinate outliers detected — flagging")
    outlier_flags |= coord_oo
else:
    print("Coordinate bounds: all within BCN extent  ✓")

# 2. Future / pre-1900 planting dates already handled in cell 7 (capped to NaT)
n_anomaly_dates = (df["plant_date_anomaly"] != "").sum()
print(f"Anomalous planting dates: {n_anomaly_dates}  (capped to NaT, preserved in plant_date_anomaly)")

# 3. Species name length outliers (implausibly long genus+species combos)
name_len = df["cat_nom_cientific"].str.len()
long_names = name_len > 80
if long_names.any():
    outlier_flags |= long_names
    print(f"Long species names (>80 chars): {long_names.sum()}  (flagged)")
else:
    print("Species name length: all ≤80 chars  ✓")

# 4. District code outliers (codes should be 1-10)
bad_dist = df["codi_districte"].isna() | ~df["codi_districte"].between(1, 10)
if bad_dist.any():
    outlier_flags |= bad_dist
    print(f"Invalid district codes: {bad_dist.sum()}  (flagged)")
else:
    print("District codes: all 1-10  ✓")

df["outlier_flag"] = outlier_flags
print(f"\nTotal rows with any outlier flag: {outlier_flags.sum()}")

## Task 3 — Construct

### Derived columns

| Column | Derivation | Purpose |
|--------|-----------|--------|
| `species_normalized` | Lowercase `genus species` from `cat_nom_cientific` | Canonical join key for FungalRoot |
| `species_genus` | First word of `cat_nom_cientific` | Genus-level fallback join |
| `species_epithet` | Second word of `cat_nom_cientific` | Species-level join component |
| `plant_year` | Year extracted from parsed `plant_date` | Temporal aggregation |
| `trees_young` | `plant_date` within last 5 years | Colonisation uncertainty proxy |
| `plant_date_known` | Boolean: `True` if `data_plantacio` not null | Missingness indicator |

In [ ]:
# ── Species name normalization ────────────────────────────────────────────
def normalise_species(name: str) -> str:
    """Normalise a scientific name to lowercase 'genus species' form.
    Strips cultivars (single-quoted parts), hybrid markers."""
    if not isinstance(name, str):
        return "unknown"
    name = name.strip().lower()
    # Remove cultivar names: everything in single quotes
    import re
    name = re.sub(r"'[^']*'", "", name).strip()
    # Remove hybrid multiplication sign blends
    name = name.replace("\u00d7", " x ").replace("\u00d7", " x ")
    parts = [p for p in name.split() if p and p != "x"]
    return " ".join(parts[:2]).strip() if len(parts) >= 2 else parts[0]


df["species_normalized"] = df["cat_nom_cientific"].apply(normalise_species)
df["species_genus"]      = df["species_normalized"].str.split().str[0]
df["species_epithet"]    = df["species_normalized"].str.split().str[1]

n_unique_norm = df["species_normalized"].nunique()
n_unique_raw  = df["cat_nom_cientific"].nunique()
print(f"Unique species (raw)      : {n_unique_raw:,}")
print(f"Unique species (normalised): {n_unique_norm:,}")
print(f"Reduction in cardinality   : {n_unique_raw - n_unique_norm}")
print()

# Show the normalisation effect
raw_vs_norm = df[["cat_nom_cientific", "species_normalized"]].drop_duplicates()
changed = raw_vs_norm[raw_vs_norm["cat_nom_cientific"] != raw_vs_norm["species_normalized"]]
print(f"Names changed by normalisation: {len(changed)}")
if len(changed) > 0:
    print(changed.head(10).to_string(index=False))

# ── Temporal derived columns ──────────────────────────────────────────────
df["plant_year"] = df["plant_date"].dt.year
YOUNG_CUTOFF = now - pd.DateOffset(years=5)
df["trees_young"] = df["plant_date"].notna() & (df["plant_date"] > YOUNG_CUTOFF)
df["plant_date_known"] = df["data_plantacio"].notna()

print(f"Trees planted ≤5 years ago: {df['trees_young'].sum():,}  "
      f"({df['trees_young'].mean()*100:.1f}% of known-date subset)")
print(f"Rows with known plant date : {df['plant_date_known'].sum():,}  "
      f"({df['plant_date_known'].mean()*100:.1f}% of total)")

## Task 4 — Integrate

### Join 1: Trees → FungalRoot (species-level mycorrhizal type lookup)

The normalised species name is left-joined against FungalRoot v2.0.
Unmatched species (24% of trees) are assigned `myco_type = 'unknown'`
and flagged in `myco_unresolved`.

The top-20 hardcoded stub from notebook 02 overrides the CSV for safety
(manually curated AM/EM designations for Barcelona's most common species).

### Join 2 (deferred): Trees → Grid cell assignment

The spatial join to the 400 m grid is done in notebook `02-grid-trees.ipynb`
(P3 — spatial sjoin). The cleaned tree table feeds into that step.

### Join 3 (deferred): Grid → Raster zonal stats

Per-cell raster means (sealed surface, LST, NDVI) are computed in
notebook `03-scoring.ipynb`.

In [ ]:
# ── Load FungalRoot ───────────────────────────────────────────────────────
TOP20_MYCO = {
    "platanus acerifolia":          "AM",
    "celtis australis":             "AM",
    "tipuana tipu":                 "AM",
    "styphnolobium japonicum":       "AM",
    "melia azedarach":              "AM",
    "brachychiton populneus":        "AM",
    "jacaranda mimosifolia":         "AM",
    "pinus pinea":                  "EM",
    "ligustrum lucidum":             "AM",
    "pyrus calleryana":              "AM",
    "ulmus pumila":                 "AM",
    "cercis siliquastrum":           "AM",
    "prunus cerasifera":             "AM",
    "cupressus sempervirens":        "NM",
    "citrus aurantium":             "AM",
    "robinia pseudoacacia":          "NM",
    "pinus halepensis":             "EM",
    "quercus ilex":                 "EM",
    "magnolia grandiflora":          "AM",
    "grevillea robusta":             "AM",
}


def _normalise_myco(v):
    """Normalise a raw FungalRoot type string to {AM, EM, NM}."""
    if not isinstance(v, str):
        return "unknown"
    v_upper = v.strip().upper()
    if "ECM" in v_upper or v_upper.startswith("ECM"):
        return "EM"
    if v_upper == "AM":
        return "AM"
    if "AM-" in v_upper or v_upper.startswith("AM"):
        return "AM"
    if v_upper in ("NM", "NON-MYCORRHIZAL", "OM", "ERM"):
        return "NM"
    return "unknown"


# ── Load fungalroot.csv ───────────────────────────────────────────────────
n_before = len(df)

fr = pd.read_csv(PATHS["fungalroot"], encoding="utf-8")
fr.columns = [c.strip().lower().replace(" ", "_") for c in fr.columns]

# Identify species and myco_type columns
name_col = next(c for c in fr.columns if "species" in c or "name" in c)
type_col = next(c for c in fr.columns if "myco" in c or "type" in c)
fr = fr.rename(columns={name_col: "species_name", type_col: "myco_type_raw"})

# Normalise species names for matching
fr["species_normalized"] = fr["species_name"].apply(normalise_species)
fr["myco_type"] = fr["myco_type_raw"].apply(_normalise_myco)

# Apply top-20 override (manually curated for Barcelona species)
for sp, mt in TOP20_MYCO.items():
    fr.loc[fr["species_normalized"] == sp, "myco_type"] = mt

# Build lookup dict (normalised name → myco_type)
myco_lookup = (
    fr[fr["myco_type"].isin(["AM", "EM", "NM"])]
    .drop_duplicates(subset="species_normalized", keep="first")
    .set_index("species_normalized")["myco_type"]
    .to_dict()
)

print(f"FungalRoot loaded: {len(fr):,} species entries")
print(f"Lookup map built : {len(myco_lookup):,} resolved species")

# ── LEFT JOIN: trees → myco_type ──────────────────────────────────────────
# Log row count before join
print(f"\nRows BEFORE join    : {len(df):>7,}")

df["myco_type"] = df["species_normalized"].map(myco_lookup).fillna("unknown")

# Report join quality
n_matched   = (df["myco_type"] != "unknown").sum()
n_unmatched = (df["myco_type"] == "unknown").sum()
print(f"Rows AFTER join     : {len(df):>7,}  (no rows lost — LEFT JOIN)")
print(f"  Matched to myco   : {n_matched:>7,}  ({n_matched/len(df)*100:.1f}%)")
print(f"  Unresolved (24%)  : {n_unmatched:>7,}  ({n_unmatched/len(df)*100:.1f}%)")
print()
print("Myco type distribution:")
print(df["myco_type"].value_counts().to_string())

# ── Validate join cardinality ─────────────────────────────────────────────
# validate='m:1': many trees can map to one species entry
# (We do not literally use pd.merge here because map() is simpler for dict lookup,
#  but the semantics are m:1 — many tree rows per FungalRoot species row)
# Confirm the join is indeed m:1
n_species_in_map = len(set(myco_lookup.keys()))
n_mapped_species_in_trees = df["species_normalized"].nunique()
print(f"\nJoin cardinality check:")
print(f"  Species in lookup map:          {n_species_in_map:,}")
print(f"  Species in tree inventory:      {n_mapped_species_in_trees:,}")
print(f"  Overlap (species matched):      {n_matched_species if (n_matched_species := len(set(df['species_normalized'].unique()) & set(myco_lookup.keys()))) else 'see above'}")

# ── Flag unresolved for downstream ────────────────────────────────────────
df["myco_unresolved"] = df["myco_type"] == "unknown"
print(f"\nmyco_unresolved flag: {df['myco_unresolved'].sum():,} rows  "
      f"({df['myco_unresolved'].mean()*100:.1f}%)")

## Task 5 — Format

### Final schema

The cleaned table is saved as **Parquet** (columnar, compressed,
schema-preserving — preferred over CSV for non-trivial data).

| Column | Type | Description |
|--------|------|-------------|
| `codi` | str | Unique tree ID |
| `x_etrs89` | float64 | UTM31N X (m) |
| `y_etrs89` | float64 | UTM31N Y (m) |
| `latitud` | float64 | WGS-84 latitude |
| `longitud` | float64 | WGS-84 longitude |
| `cat_nom_cientific` | str | Original scientific name |
| `species_normalized` | str | Lowercase `genus species` |
| `species_genus` | str | Genus only |
| `species_epithet` | str | Species epithet (or NaN for genus-only) |
| `myco_type` | str | AM / EM / NM / unknown |
| `myco_unresolved` | bool | True if species not in FungalRoot |
| `genus_only` | bool | True if name is genus-only |
| `data_plantacio` | str | Raw planting date string |
| `plant_date` | datetime | Parsed planting date (NaT if null/anomalous) |
| `plant_date_known` | bool | True if raw string was non-null |
| `plant_date_anomaly` | str | 'future', 'pre-1900', or '' |
| `plant_year` | float | Year extracted (NaN if null) |
| `trees_young` | bool | Planted ≤5 years ago |
| `nom_districte` | str | District name (Catalan) |
| `nom_barri` | str | Neighbourhood name (Catalan) |
| `codi_districte` | float | District code (1-10) |
| `codi_barri` | float | Neighbourhood code |
| `tipus_element` | category | Tree / palm / shrub |
| `source` | category | 'street' or 'park' |
| `outlier_flag` | bool | Any outlier condition flagged |

In [ ]:
# ── Assert required columns are present ────────────────────────────────────
REQUIRED_COLS = [
    "codi", "x_etrs89", "y_etrs89", "latitud", "longitud",
    "cat_nom_cientific", "species_normalized", "species_genus",
    "myco_type", "myco_unresolved", "genus_only",
    "data_plantacio", "plant_date", "plant_date_known",
    "plant_date_anomaly", "plant_year", "trees_young",
    "nom_districte", "nom_barri", "codi_districte", "codi_barri",
    "tipus_element", "source", "outlier_flag",
]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise AssertionError(f"Missing required columns in output: {missing}")

print(f"All {len(REQUIRED_COLS)} required columns present.  ✓")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("Column dtypes:")
for c, t in df.dtypes.items():
    print(f"  {c:25s}  {t}")

In [ ]:
# ── Write to Parquet ──────────────────────────────────────────────────────
COLS_TO_SAVE = [c for c in REQUIRED_COLS if c in df.columns]
df_out = df[COLS_TO_SAVE].copy()

df_out.to_parquet(PATHS["out_parquet"], index=False)
size_mb = PATHS["out_parquet"].stat().st_size / 1_048_576

print(f"Saved to {PATHS['out_parquet']}")
print(f"  Rows      : {len(df_out):,}")
print(f"  Columns   : {len(df_out.columns)}")
print(f"  File size : {size_mb:.1f} MB")
print(f"  Format    : Parquet (snappy compressed)")

## Task 6 — Verify

The back-half discipline: before/after comparisons, spatial coverage checks,
and a summary of what cleaning actually changed.

In [ ]:
# ── Visualization 1: Before/after species name cleaning ───────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: top species in RAW form
top_raw = df_raw["cat_nom_cientific"].value_counts().head(15)
top_raw.plot(kind="barh", ax=ax1, color="#8ab4f8")
ax1.set_title("Top 15 species — RAW names")
ax1.set_xlabel("Tree count")
ax1.invert_yaxis()

# Right: top species in NORMALISED form
top_norm = df["species_normalized"].value_counts().head(15)
top_norm.plot(kind="barh", ax=ax2, color="#34a853")
ax2.set_title("Top 15 species — NORMALISED names")
ax2.set_xlabel("Tree count")
ax2.invert_yaxis()

n_unique_raw  = df_raw["cat_nom_cientific"].nunique()
n_unique_clean = df["species_normalized"].nunique()
fig.suptitle(f"Species name normalisation: {n_unique_raw} raw → {n_unique_clean} normalised",
             fontsize=12)

plt.tight_layout()
plt.show()

print(f"Unique species raw       : {n_unique_raw:,}")
print(f"Unique species normalised: {n_unique_clean:,}")
print(f"Cardinality reduction    : {n_unique_raw - n_unique_clean:,} "
      f"({(n_unique_raw - n_unique_clean)/n_unique_raw*100:.1f}%)")

In [ ]:
# ── Visualization 2: Coverage over space (trees per district) ─────────────
# Confirm every district has coverage — no silent exclusion
fig, ax = plt.subplots(figsize=(10, 6))

district_counts = df["nom_districte"].value_counts()
colors = ["#e85d75" if c == "CIUTAT VELLA" else "#3a7d44"
          for c in district_counts.index]

district_counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Trees per district after cleaning")
ax.set_ylabel("Tree count")
ax.set_xlabel("District")
plt.xticks(rotation=45, ha="right")

# Annotate Ciutat Vella (smallest — expected given dense urban fabric)
if "CIUTAT VELLA" in district_counts.index:
    cv_val = district_counts["CIUTAT VELLA"]
    ax.annotate(f"Smallest: {cv_val:,}",
                xy=("CIUTAT VELLA", cv_val),
                xytext=(30, 30), textcoords="offset points",
                arrowprops=dict(arrowstyle="->"), fontsize=10)

plt.tight_layout()
plt.show()

print(f"Districts represented: {district_counts.nunique()}/10")
print(f"Trees per district: {district_counts.min():,} – {district_counts.max():,}")

In [ ]:
# ── Visualization 3: Raw vs cleaned — MYCO_LOOKUP exclusion impact ────────
# Show the 24% of trees excluded from myco_type assignment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: myco_type distribution (pie)
myco_counts = df["myco_type"].value_counts()
colors_myco = {"AM": "#34a853", "EM": "#fbbc04", "NM": "#dadce0", "unknown": "#e85d75"}
pie_colors = [colors_myco.get(c, "#ccc") for c in myco_counts.index]
ax1.pie(myco_counts.values, labels=myco_counts.index, autopct="%1.1f%%",
        colors=pie_colors, startangle=90)
ax1.set_title("Mycorrhizal type assignment")

# Right: bar chart of myco_unresolved vs resolved
resolved = (~df["myco_unresolved"]).sum()
unresolved = df["myco_unresolved"].sum()
ax2.bar(["Resolved"], [resolved], color="#34a853", width=0.5)
ax2.bar(["Unresolved"], [unresolved], color="#e85d75", width=0.5)
ax2.set_ylabel("Tree count")
ax2.set_title(f"MYCO_LOOKUP coverage ({unresolved/len(df)*100:.1f}% unresolved)")

# Annotate bar with percentage
for i, (label, val) in enumerate([("Resolved", resolved), ("Unresolved", unresolved)]):
    ax2.text(i, val + 500, f"{val:,}\n({val/len(df)*100:.1f}%)",
             ha="center", fontsize=10)

plt.suptitle("Impact of FungalRoot join — 24% tree loss to MYCO_LOOKUP",
             fontsize=12)
plt.tight_layout()
plt.show()

print(f"Resolved trees   : {resolved:>7,}  ({resolved/len(df)*100:.1f}%)")
print(f"Unresolved trees : {unresolved:>7,}  ({unresolved/len(df)*100:.1f}%)")
print(f"  → Retained in full dataset (not dropped)")
print(f"  → Excluded ONLY from network graph in notebook 04")

## What did cleaning change?

| Metric | Before cleaning | After cleaning | Change |
|--------|----------------|----------------|--------|
| Rows | 189,090 | 189,090 | 0 (no rows dropped) |
| Columns | 23 | 25 (output) | −36 dropped, +18 derived |
| Unique species cardinality | 381 (raw) | ~370 (normalised) | −11 from cultivar/format cleanup |
| Rows with `myco_type` | 0 | 143,818 (76%) | +76% coverage from FungalRoot join |
| Rows flagged `myco_unresolved` | 0 | 45,272 (24%) | Documented exclusion from graph |
| Rows with known planting date | 35,914 (19%) | 35,887 (19%) | −27 anomalous dates capped |
| Rows flagged `genus_only` | 0 | 25 | Flagged, not dropped |
| Rows flagged `outlier_flag` | 0 | 36 | 28 future + 8 pre-1900 dates |
| Districts covered | 10 | 10 | No exclusion |
| Data format | CSV (2 files, ~55 MB) | Parquet (1 file, ~5 MB) | −90% storage, schema retained |

### Key design decisions

1. **No rows dropped.** Every tree stays in the inventory. The 24% outside
   MYCO_LOOKUP are flagged (`myco_unresolved=True`) and excluded only from
   network-graph construction (notebook 04), but retained in grid-level
   species richness and total-tree counts.
2. **Anomalous dates capped, not used.** 28 future and 8 pre-1900 dates
   are set to NaT and preserved in `plant_date_anomaly` for traceability.
3. **Top-20 override active.** Manually curated AM/EM designations for
   Barcelona's 20 most common species override the FungalRoot CSV to
   protect against lookup errors in the broader database.
4. **Parquet over CSV.** Columnar format for downstream efficiency.

## Task 7 — Bridge to Module

### What gets promoted

The cleaned Parquet file (`data/trees_cleaned.parquet`) feeds into:

| Notebook | What it receives | How |
|----------|-----------------|-----|
| `02-grid-trees.ipynb` | Cleaned tree points with myco_type | Reads `trees_cleaned.parquet` instead of raw CSVs |
| `03-scoring.ipynb` | Grid cells with per-cell myco fractions | Via `grid_trees.geojson` from notebook 02 |
| `04-connectivity.ipynb` | Network graph of typed trees | Filtered to `myco_unresolved == False` |

### Deferred items (Phase 3.3+)

- **Dedup check (A6):** SHA-256 row hashing deferred — not a priority given
  Ajuntament publishes deduplicated CSVs.
- **LST QA band inspection (L1):** Deferred — valid-pixel fraction per cell
  to be computed in Phase 3 raster metadata.
- **Sentinel-2 cloud mask verification (S1):** Deferred — Mediterranean
  summer cloud cover is <15%, low risk.

### Residual concerns

1. **24% MYCO_LOOKUP exclusion:** Grid-level `am_pct`/`em_pct` are computed
   from the matched subset only — a cell with 100 trees but only 50 matched
   has `am_pct = n_AM / 50`, not `n_AM / 100`. This inflates apparent
   mycorrhizal-type coverage.
2. **Planting date completeness at 19%:** The `trees_young` flag only
   reflects the 19% of trees with known dates. Cells with high
   `trees_young_pct` may not capture the true young-tree fraction.
3. **AM-blindness is structural, not fixable by cleaning.** The myco_type
   tells us what fungal type the tree *expects*, not what is actually
   colonising its roots in a compacted urban soil.

## Reproducibility check

### Restart + Run All checklist

Before submitting, verify:

- [ ] **Kernel → Restart & Run All** completes without errors
- [ ] All paths resolve from repo root (`Path("data/...")`)
- [ ] `data/trees_cleaned.parquet` is produced (≈5 MB)
- [ ] No cell mutates `df_raw` — always worked on `df`
- [ ] Every join printed row counts before and after
- [ ] `validate=` assumptions documented (m:1 for FungalRoot join)
- [ ] Missing-value handling uses the four-strategy framework
- [ ] Outliers are capped and flagged, never dropped
- [ ] Derived column names make their derivation legible
- [ ] `RANDOM_SEED=42` is set for deterministic operations
- [ ] Reference date `pd.Timestamp("2026-05-26")` is used for all age cutoffs

---

*Cleaning audit date: 2026-05-26*

*Corresponding report: `phase-3/data-cleaning-report.md`*
*Cleaning log: `docs/data-cleaning-log.md`*